# One-dimensional free-electron Bloch bands

**Status:** Maintained calculated results for the idealized bare free-electron model. This is not an interacting electron, crystal-potential, or material-specific band structure.

**Prerequisite:** [One-dimensional free-electron Bloch representation](../../../../docs/lecturenotes/solidstate/bloch1d/freeelectron/index.md)

The physical model belongs to `solidstate.bloch1d.freeelectron`. The endpoint-excluded finite-difference grid and Bloch-twisted seam belong independently to `periodic.pbc1d`.


## Free electron represented through Bloch fibers

The underlying free electron has continuous physical wave number $q$ and energy

$$E(q)=\frac{\hbar^2q^2}{2m_e}.$$

Representing the infinite line with a reference cell of length $a$ writes

$$q_n=k+\frac{2\pi n}{a},\qquad k\in[-\pi/a,\pi/a].$$

For fixed $k$, the cell satisfies $\psi(x+a)=e^{ika}\psi(x)$. At $k=0$ this fixed fiber is mathematically equivalent to periodic boundary conditions on a ring, but the complete family of $k$ fibers represents the free electron on the infinite line with folded bands.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.solidstate.bloch1d.cell import BlochPrimitiveCell1D
from projectkoios.physkit.solidstate.bloch1d.freeelectron.finite_difference import FreeElectronBloch1DFiniteDifferenceHamiltonianConstructor, FreeElectronBloch1DFiniteDifferenceHamiltonianRequest
from projectkoios.physkit.solidstate.bloch1d.freeelectron.model import FreeElectronBloch1D
from projectkoios.physkit.units import PhysicalUnit, ScalarQuantity, UnitSystem

cell = BlochPrimitiveCell1D(
    oriented_length=ScalarQuantity(10.0, UnitSystem.METAL.length_unit),
    unit_system=UnitSystem.METAL,
)
model = FreeElectronBloch1D(cell=cell)
reciprocal_unit = PhysicalUnit("1 / angstrom")


## Exact folded bands

The band index below is an ordering of reciprocal modes, not evidence of bonding, localization, or a lattice potential.


In [ ]:
mode_indices = tuple((n,) for n in range(-4, 5))
k_values = np.linspace(-np.pi / cell.length.magnitude, np.pi / cell.length.magnitude, 301)
exact_rows = []
for k in k_values:
    spectrum = model.evaluate_modes(
        mode_indices=mode_indices,
        bloch_wave_number=ScalarQuantity(float(k), reciprocal_unit),
    )
    exact_rows.append(np.sort(spectrum.energies.magnitude)[:5])
exact_bands = np.asarray(exact_rows)
fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.plot(k_values * cell.length.magnitude / np.pi, exact_bands)
ax.set(xlabel=r"$k/(\pi/a)$", ylabel="Energy (eV)", title="Exact folded free-electron bands")
ax.grid(alpha=0.25)
fig.tight_layout()


## Independent `pbc1d` finite-difference route

The numerical route constructs a centered second derivative on an endpoint-excluded cell. Only the two seam entries carry the Bloch phase. Its error is a discretization error and remains distinct from the exact free-electron model.


In [ ]:
sample_k = np.linspace(-np.pi / cell.length.magnitude, np.pi / cell.length.magnitude, 81)
fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.2), constrained_layout=True)
for ax, point_count in zip(axes, (16, 32), strict=True):
    numerical_rows = []
    for k in sample_k:
        represented = FreeElectronBloch1DFiniteDifferenceHamiltonianConstructor().action(
            request=FreeElectronBloch1DFiniteDifferenceHamiltonianRequest(
                model=model,
                point_count=point_count,
                bloch_wave_number=ScalarQuantity(float(k), reciprocal_unit),
            )
        )
        numerical_rows.append(np.linalg.eigvalsh(represented.represented_hamiltonian.to_csr().toarray())[:5])
    numerical_bands = np.asarray(numerical_rows)
    exact_at_samples = []
    for k in sample_k:
        row = model.evaluate_modes(
            mode_indices=mode_indices,
            bloch_wave_number=ScalarQuantity(float(k), reciprocal_unit),
        ).energies.magnitude
        exact_at_samples.append(np.sort(row)[:5])
    ax.plot(sample_k * cell.length.magnitude / np.pi, np.asarray(exact_at_samples), color="black", linewidth=1.0)
    ax.plot(sample_k * cell.length.magnitude / np.pi, numerical_bands, linestyle="--", linewidth=0.9)
    ax.set(title=f"{point_count} endpoint-excluded points", xlabel=r"$k/(\pi/a)$", ylabel="Energy (eV)")
    ax.grid(alpha=0.2)


## Numerical Bloch state

The finite-difference eigenvector is normalized with the grid measure $\Delta x$. A free-electron state remains extended and has uniform probability density.


In [ ]:
k_selected = 0.37 * np.pi / cell.length.magnitude
represented = FreeElectronBloch1DFiniteDifferenceHamiltonianConstructor().action(
    request=FreeElectronBloch1DFiniteDifferenceHamiltonianRequest(
        model=model,
        point_count=64,
        bloch_wave_number=ScalarQuantity(float(k_selected), reciprocal_unit),
    )
)
energies, states = np.linalg.eigh(represented.represented_hamiltonian.to_csr().toarray())
state = states[:, 0]
dx = represented.grid.spacing.magnitude
state = state / np.sqrt(np.sum(np.abs(state) ** 2) * dx)
x = represented.grid.positions.magnitude
assert np.isclose(np.sum(np.abs(state) ** 2) * dx, 1.0)
fig, ax = plt.subplots(figsize=(7.2, 4.5))
ax.plot(x, state.real, label=r"$\mathrm{Re}\,\psi$")
ax.plot(x, state.imag, label=r"$\mathrm{Im}\,\psi$")
ax.plot(x, np.abs(state) ** 2, label=r"$|\psi|^2$")
ax.set(xlabel="x (Å)", ylabel="Amplitude / density", title=f"Lowest represented state, E={energies[0]:.5f} eV")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()


## Fourier content

The dominant discrete Fourier component identifies the reciprocal contribution to $q=k+G_n$. Degenerate or nearly degenerate numerical eigenspaces may be returned in arbitrary orthonormal combinations.


In [ ]:
coefficients = np.fft.fftshift(np.fft.fft(state))
weights = np.abs(coefficients) ** 2
weights = weights / np.sum(weights)
reciprocal_indices = np.fft.fftshift(np.fft.fftfreq(state.size, d=1.0 / state.size))
fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.stem(reciprocal_indices, weights)
ax.set(xlabel="Reciprocal index n", ylabel="Fourier weight", title="Fourier content of the represented state", xlim=(-8, 8))
ax.grid(alpha=0.25)
fig.tight_layout()


## Interpretation and exercises

The exact route verifies the folded free-electron model. The `pbc1d` route independently verifies its finite-difference representation and exposes grid error. Neither route contains a crystal potential.

1. Demonstrate second-order convergence of the lowest represented energy.
2. Compare the $k=0$ periodic fiber with a nonzero twisted fiber.
3. Add a periodic potential as a distinct Bloch model rather than modifying the free-electron contract.
4. Explain why inspecting individual eigenvectors is unreliable inside a degenerate eigenspace.
5. Compare the exact centered-difference symbol with the continuum dispersion.
